# Operations Manager Interview Scheduling

## Section 1: Imports

In [1]:
import pandas as pd
from datetime import datetime, timedelta, time
import collections
import re # For parsing constraints

# Suppress pandas FutureWarnings
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

## Section 2: Read Excel and Preview DataFrame

In [2]:
# Read the Excel file into a pandas DataFrame
df_candidates = pd.read_excel('/content/interview_candidates.xlsx')

# Display the first 5 rows of the DataFrame to preview the data
print("Candidate Data Preview:")
display(df_candidates.head())

Candidate Data Preview:


,candidate_id,name,availability_constraint
0,S01,Ajay Mathur,NaN
1,S02,Bhargavi N,NaN
2,S03,Chandana Rao,Available only after 3:00 PM
3,S04,Dinesh Kotta,NaN
4,S05,Esha Bhatt,Second-round: must be interviewed by Priya (sa...


## Section 3: Create Interviewer Slots and Definitions

In [3]:
# --- Global Definitions ---
interview_duration = timedelta(minutes=30)

# Define valid start times for interviews as strings
valid_start_times_str = [
    '10:00', '10:30', '11:00', '11:30', '12:00',
    '13:30', '14:00', '14:30', '15:00', '15:30', '16:00'
]
# Convert valid start times to datetime.time objects for easier comparison
valid_start_times_obj = [datetime.strptime(ts, '%H:%M').time() for ts in valid_start_times_str]

# Define interviewer availabilities as (start_time_str, end_time_str) tuples
priya_availability = [('10:00', '12:30'), ('15:00', '16:00')]
rahul_availability = [('13:30', '16:30')]

# --- Helper Functions for Slot Generation ---

def parse_time_str_to_time(time_str):
    """Converts a 'HH:MM' string to a datetime.time object."""
    return datetime.strptime(time_str, '%H:%M').time()

def time_to_str(time_obj):
    """Converts a datetime.time object to a 'HH:MM' string."""
    return time_obj.strftime('%H:%M')

def generate_slots_for_interval(start_interval_str, end_interval_str, valid_start_times):
    """
    Generates all 30-minute interview slots within a given interval,
    respecting predefined valid start times.
    Returns a list of (start_time, end_time) tuples (datetime.time objects).
    """
    slots = []
    current_dt = datetime.combine(datetime.min, parse_time_str_to_time(start_interval_str))
    end_dt = datetime.combine(datetime.min, parse_time_str_to_time(end_interval_str))

    while current_dt + interview_duration <= end_dt:
        if current_dt.time() in valid_start_times:
            slots.append((current_dt.time(), (current_dt + interview_duration).time()))
        current_dt += interview_duration
    return slots

def get_interviewer_slots(availability_intervals, valid_start_times):
    """
    Aggregates all 30-minute interview slots for an interviewer from their availability intervals.
    Returns a list of (start_time, end_time) tuples.
    """
    all_slots = []
    for start_str, end_str in availability_intervals:
        all_slots.extend(generate_slots_for_interval(start_str, end_str, valid_start_times))
    return all_slots

def generate_rahul_break_scenarios(rahul_full_availability_intervals, valid_start_times):
    """
    Generates all possible Rahul schedules, each with a different 30-minute break slot.
    Returns a list of dictionaries, where each dict contains:
    - 'break_slot': (start_time, end_time) of the break
    - 'interview_slots': list of remaining interview slots
    """
    rahul_all_possible_slots = get_interviewer_slots(rahul_full_availability_intervals, valid_start_times)
    break_scenarios = []

    # Each possible interview slot can be a break slot
    for break_idx, potential_break_slot in enumerate(rahul_all_possible_slots):
        remaining_interview_slots = [
            slot for idx, slot in enumerate(rahul_all_possible_slots) if idx != break_idx
        ]
        break_scenarios.append({
            'break_slot': potential_break_slot,
            'interview_slots': remaining_interview_slots
        })
    return break_scenarios

# --- Generate Interviewer Slots ---
priya_slots_raw = get_interviewer_slots(priya_availability, valid_start_times_obj)

# Generate all Rahul break scenarios
rahul_break_scenarios = generate_rahul_break_scenarios(rahul_availability, valid_start_times_obj)

print("Priya's available slots:", [f'{time_to_str(s)}-{time_to_str(e)}' for s,e in priya_slots_raw])
print(f"Generated {len(rahul_break_scenarios)} possible break scenarios for Rahul.")

Priya's available slots: ['10:00-10:30', '10:30-11:00', '11:00-11:30', '11:30-12:00', '12:00-12:30', '15:00-15:30', '15:30-16:00']
Generated 6 possible break scenarios for Rahul.


## Section 4: Parse Candidate Constraints

In [4]:
def parse_candidate_constraint(constraint_str):
    """
    Parses a single candidate constraint string into a structured dictionary.
    Returns a dictionary with keys like 'interviewer_must_be', 'after_time', 'before_time',
    'unavailable_periods' (list of (start_time, end_time) tuples).
    """
    constraints = {
        'interviewer_must_be': None,
        'after_time': None,
        'before_time': None,
        'unavailable_periods': []
    }
    if pd.isna(constraint_str) or constraint_str == '':
        return constraints

    # Must be interviewed by [Interviewer]
    match_interviewer = re.search(r'Must be interviewed by (Priya|Rahul)', constraint_str)
    if match_interviewer:
        constraints['interviewer_must_be'] = match_interviewer.group(1)

    # Available only after HH:MM [AM/PM] / after HH:MM
    match_after = re.search(r'Available only after (\d{1,2}:\d{2}) (AM|PM)?', constraint_str)
    if match_after:
        time_str = match_after.group(1)
        ampm = match_after.group(2)
        if ampm: # Convert 12-hour to 24-hour
            t = datetime.strptime(f'{time_str} {ampm}', '%I:%M %p').time()
        else:
            t = datetime.strptime(time_str, '%H:%M').time()
        constraints['after_time'] = t

    # Interview must end by HH:MM [AM/PM] / by HH:MM
    match_before = re.search(r'Interview must end by (\d{1,2}:\d{2}) (AM|PM)?', constraint_str)
    if match_before:
        time_str = match_before.group(1)
        ampm = match_before.group(2)
        if ampm: # Convert 12-hour to 24-hour
            t = datetime.strptime(f'{time_str} {ampm}', '%I:%M %p').time()
        else:
            t = datetime.strptime(time_str, '%H:%M').time()
        constraints['before_time'] = t

    # Unavailable between HH:MM [AM/PM] and HH:MM [AM/PM]
    match_unavailable = re.search(r'Unavailable between (\d{1,2}:\d{2}) (AM|PM)? and (\d{1,2}:\d{2}) (AM|PM)?', constraint_str)
    if match_unavailable:
        start_time_str = match_unavailable.group(1)
        start_ampm = match_unavailable.group(2)
        end_time_str = match_unavailable.group(3)
        end_ampm = match_unavailable.group(4)

        if start_ampm: start_t = datetime.strptime(f'{start_time_str} {start_ampm}', '%I:%M %p').time()
        else: start_t = datetime.strptime(start_time_str, '%H:%M').time()
        if end_ampm: end_t = datetime.strptime(f'{end_time_str} {end_ampm}', '%I:%M %p').time()
        else: end_t = datetime.strptime(end_time_str, '%H:%M').time()

        constraints['unavailable_periods'].append((start_t, end_t))

    return constraints


# Apply the parsing function to each candidate
df_candidates['parsed_constraints'] = df_candidates['availability_constraint'].apply(parse_candidate_constraint)

print("Parsed Constraints for Candidates (first 5):")
for idx, row in df_candidates.head().iterrows():
    print(f"Candidate {row['candidate_id']}: {row['parsed_constraints']}")

Parsed Constraints for Candidates (first 5):
Candidate S01: {'interviewer_must_be': None, 'after_time': None, 'before_time': None, 'unavailable_periods': []}
Candidate S02: {'interviewer_must_be': None, 'after_time': None, 'before_time': None, 'unavailable_periods': []}
Candidate S03: {'interviewer_must_be': None, 'after_time': datetime.time(15, 0), 'before_time': None, 'unavailable_periods': []}
Candidate S04: {'interviewer_must_be': None, 'after_time': None, 'before_time': None, 'unavailable_periods': []}
Candidate S05: {'interviewer_must_be': None, 'after_time': None, 'before_time': None, 'unavailable_periods': []}


## Section 5: Scheduling Algorithm

In [9]:
def check_slot_compatibility(slot_start, slot_end, constraints):
    """
    Checks if a given slot (start_time, end_time) is compatible with candidate constraints.
    Returns True if compatible, False otherwise.
    """
    if constraints['after_time'] and slot_start < constraints['after_time']:
        return False, "Slot is before required start time."

    if constraints['before_time'] and slot_end > constraints['before_time']:
        return False, "Slot is after required end time."

    for unavailable_start, unavailable_end in constraints['unavailable_periods']:
        # Check for overlap: [slot_start, slot_end) and [unavailable_start, unavailable_end)
        if not (slot_end <= unavailable_start or slot_start >= unavailable_end):
            return False, "Slot overlaps with an unavailable period."

    return True, "Compatible."

def find_best_schedule_for_scenario(
    candidates_df,
    priya_available_slots_initial,
    rahul_available_slots_initial,
    rahul_break_slot
):
    """
    Attempts to schedule candidates for a given Rahul break scenario.
    Returns a tuple: (scheduled_count, schedule_details, unscheduled_notes, priya_final_slots, rahul_final_slots)
    """
    # Deep copy slots to avoid modifying the original lists for this scenario
    priya_current_slots = list(priya_available_slots_initial)
    rahul_current_slots = list(rahul_available_slots_initial)

    # Store assignments for this scenario
    current_schedule_details = [] # [{'candidate_id', 'interviewer', 'start_time', 'end_time', 'notes'}]
    unscheduled_notes = {}
    scheduled_candidate_ids = set()

    # Sort candidates for consistent greedy assignment (e.g., by ID)
    sorted_candidates = candidates_df.sort_values(by='candidate_id').to_dict('records')

    # --- Phase 1: Assign candidates with specific interviewer constraints ---
    for candidate in sorted_candidates:
        if candidate['candidate_id'] in scheduled_candidate_ids: continue

        constraints = candidate['parsed_constraints']
        interviewer_pref = constraints.get('interviewer_must_be')

        if interviewer_pref:
            target_interviewer_slots = []
            if interviewer_pref == 'Priya':
                target_interviewer_slots = priya_current_slots
            elif interviewer_pref == 'Rahul':
                target_interviewer_slots = rahul_current_slots

            assigned = False
            # Try to find the earliest compatible slot
            for slot_idx, (s_time, e_time) in enumerate(target_interviewer_slots):
                is_compatible, reason = check_slot_compatibility(s_time, e_time, constraints)
                if is_compatible:
                    current_schedule_details.append({
                        'candidate_id': candidate['candidate_id'],
                        'interviewer': interviewer_pref,
                        'start_time': s_time,
                        'end_time': e_time,
                        'notes': ''
                    })
                    scheduled_candidate_ids.add(candidate['candidate_id'])
                    del target_interviewer_slots[slot_idx] # Remove the assigned slot
                    assigned = True
                    break
            if not assigned:
                unscheduled_notes[candidate['candidate_id']] = f"No compatible slot with {interviewer_pref} ({reason})."

    # --- Phase 2: Assign remaining candidates (greedy) ---
    for candidate in sorted_candidates:
        if candidate['candidate_id'] in scheduled_candidate_ids: continue

        constraints = candidate['parsed_constraints']
        assigned = False
        fail_reason = "No compatible interview slot available."

        # Try Priya first
        for slot_idx, (s_time, e_time) in enumerate(priya_current_slots):
            is_compatible, reason = check_slot_compatibility(s_time, e_time, constraints)
            if is_compatible:
                current_schedule_details.append({
                    'candidate_id': candidate['candidate_id'],
                    'interviewer': 'Priya',
                    'start_time': s_time,
                    'end_time': e_time,
                    'notes': ''
                })
                scheduled_candidate_ids.add(candidate['candidate_id'])
                del priya_current_slots[slot_idx]
                assigned = True
                break
        if assigned: continue

        # Try Rahul if not assigned by Priya
        for slot_idx, (s_time, e_time) in enumerate(rahul_current_slots):
            is_compatible, reason = check_slot_compatibility(s_time, e_time, constraints)
            if is_compatible:
                current_schedule_details.append({
                    'candidate_id': candidate['candidate_id'],
                    'interviewer': 'Rahul',
                    'start_time': s_time,
                    'end_time': e_time,
                    'notes': ''
                })
                scheduled_candidate_ids.add(candidate['candidate_id'])
                del rahul_current_slots[slot_idx]
                assigned = True
                break

        if not assigned:
            # If still not assigned, record as unscheduled
            if candidate['candidate_id'] not in unscheduled_notes:
                # More specific reason if known from check_slot_compatibility
                temp_slots = priya_available_slots_initial + rahul_available_slots_initial # All possible slots
                reasons = set()
                for s_time, e_time in temp_slots:
                    _, reason = check_slot_compatibility(s_time, e_time, constraints)
                    if reason != "Compatible.":
                        reasons.add(reason)

                if reasons: # if any specific reasons were found
                    fail_reason = "; ".join(list(reasons))
                else:
                    fail_reason = "No compatible interview slot available."

                unscheduled_notes[candidate['candidate_id']] = fail_reason

    return (
        len(scheduled_candidate_ids),
        current_schedule_details,
        unscheduled_notes,
        priya_current_slots, # Remaining slots for Priya
        rahul_current_slots, # Remaining slots for Rahul
        rahul_break_slot # The break that was chosen
    )

# --- Main Scheduling Execution ---

max_scheduled_count = -1
best_schedule_details = None
best_unscheduled_notes = None
best_priya_final_slots = None
best_rahul_final_slots = None
best_rahul_break = None

print(f"Attempting to find the optimal schedule across {len(rahul_break_scenarios)} Rahul break scenarios...")

for scenario in rahul_break_scenarios:
    rahul_interview_slots_for_scenario = scenario['interview_slots']
    rahul_current_break_slot = scenario['break_slot']

    (scheduled_count, schedule_details, unscheduled_notes,
    priya_rem_slots, rahul_rem_slots, rahul_break_chosen) = find_best_schedule_for_scenario(
        df_candidates,
        priya_slots_raw, # Pass raw slots, function copies them
        rahul_interview_slots_for_scenario, # Rahul's slots for THIS scenario
        rahul_current_break_slot
    )

    if scheduled_count > max_scheduled_count:
        max_scheduled_count = scheduled_count
        best_schedule_details = schedule_details
        best_unscheduled_notes = unscheduled_notes
        best_priya_final_slots = priya_rem_slots
        best_rahul_final_slots = rahul_rem_slots
        best_rahul_break = rahul_break_chosen

print(f"Optimization complete. Maximum scheduled candidates: {max_scheduled_count} out of {len(df_candidates)}.")

Attempting to find the optimal schedule across 6 Rahul break scenarios...
Optimization complete. Maximum scheduled candidates: 12 out of 14.


## Section 6: Generate Output DataFrame

In [10]:
output_df_data = []

# Add scheduled candidates to the output data
for item in best_schedule_details:
    output_df_data.append({
        'candidate_id': item['candidate_id'],
        'interviewer': item['interviewer'],
        'start_time': time_to_str(item['start_time']),
        'end_time': time_to_str(item['end_time']),
        'notes': item['notes']
    })

# Add unscheduled candidates to the output data
for candidate_id in df_candidates['candidate_id']:
    if candidate_id not in [d['candidate_id'] for d in best_schedule_details]:
        output_df_data.append({
            'candidate_id': candidate_id,
            'interviewer': 'NONE',
            'start_time': '',
            'end_time': '',
            'notes': best_unscheduled_notes.get(candidate_id, "No compatible interview slot available.")
        })

# Create the final output DataFrame
output_df = pd.DataFrame(output_df_data)

# Ensure all candidates are present and sort by candidate_id for consistent output
output_df['candidate_id'] = output_df['candidate_id'].astype(df_candidates['candidate_id'].dtype) # Match original type
output_df = output_df.set_index('candidate_id').reindex(df_candidates['candidate_id']).reset_index()

print("Generated Output DataFrame Preview:")
display(output_df.head(10)) # Display more rows for context

# Check if all candidates are included
assert len(output_df) == len(df_candidates), "Mismatch in total candidates in output DataFrame!"

Generated Output DataFrame Preview:


,candidate_id,interviewer,start_time,end_time,notes
0,S01,Priya,10:00,10:30,
1,S02,Priya,10:30,11:00,
2,S03,Priya,15:00,15:30,
3,S04,Priya,11:00,11:30,
4,S05,Priya,11:30,12:00,
5,S06,Priya,12:00,12:30,
6,S07,Priya,15:30,16:00,
7,S08,Rahul,14:00,14:30,
8,S09,Rahul,14:30,15:00,
9,S10,Rahul,15:00,15:30,


## Section 7: Save CSV

In [11]:
# Create the output directory if it doesn't exist
import os
os.makedirs('output', exist_ok=True)

# Save the output DataFrame to a CSV file
output_csv_path = 'output/task2.csv'
output_df.to_csv(output_csv_path, index=False)
print(f"Scheduling results saved to '{output_csv_path}'")

Scheduling results saved to 'output/task2.csv'


## Section 8: Summary

In [12]:
print("--- Scheduling Summary ---")
print(f"Total Candidates: {len(df_candidates)}")
print(f"Scheduled Candidates: {max_scheduled_count}")
print(f"Unscheduled Candidates: {len(df_candidates) - max_scheduled_count}")

print("\n--- Interviewer Schedules ---")

priya_scheduled = output_df[output_df['interviewer'] == 'Priya']
rahul_scheduled = output_df[output_df['interviewer'] == 'Rahul']

print("Priya's Schedule:")
if not priya_scheduled.empty:
    display(priya_scheduled[['candidate_id', 'start_time', 'end_time']])
else:
    print("No candidates scheduled for Priya.")

print("\nRahul's Schedule:")
if not rahul_scheduled.empty:
    display(rahul_scheduled[['candidate_id', 'start_time', 'end_time']])
else:
    print("No candidates scheduled for Rahul.")

print("\nRahul's Break:")
if best_rahul_break:
    print(f"Rahul's 30-minute break is from {time_to_str(best_rahul_break[0])} to {time_to_str(best_rahul_break[1])}.")
else:
    print("Rahul's break could not be determined (no optimal schedule found). This should not happen with valid input.")

print("\n--- Unscheduled Candidates Details ---")
unscheduled_df = output_df[output_df['interviewer'] == 'NONE']
if not unscheduled_df.empty:
    display(unscheduled_df[['candidate_id', 'notes']])
else:
    print("All candidates were scheduled!")

--- Scheduling Summary ---
Total Candidates: 14
Scheduled Candidates: 12
Unscheduled Candidates: 2

--- Interviewer Schedules ---
Priya's Schedule:


,candidate_id,start_time,end_time
0,S01,10:00,10:30
1,S02,10:30,11:00
2,S03,15:00,15:30
3,S04,11:00,11:30
4,S05,11:30,12:00
5,S06,12:00,12:30
6,S07,15:30,16:00



Rahul's Schedule:


,candidate_id,start_time,end_time
7,S08,14:00,14:30
8,S09,14:30,15:00
9,S10,15:00,15:30
10,S11,15:30,16:00
11,S12,16:00,16:30



Rahul's Break:
Rahul's 30-minute break is from 13:30 to 14:00.

--- Unscheduled Candidates Details ---


,candidate_id,notes
12,S13,No compatible interview slot available.
13,S14,No compatible interview slot available.
